Adzuna API exploration (Day 4) → becomes daily ingestion (Day 5).
Reads:  Adzuna /jobs/us/search endpoint
Writes: (Day 5) jobmarket.bronze.job_postings_api

## Adzuna → Silver field map
| Adzuna JSON            | Silver column      | Notes                        |
|------------------------|--------------------|------------------------------|
| title                  | title_raw          |                              |
| company.display_name   | company            | nested object                |
| location.area          | city, state        | array: [country, state, city]|
| salary_min/salary_max  | salary_min/max     | annual USD already?  verify  |
| salary_is_predicted    | salary_is_estimated| "0"/"1" as string? verify    |
| description            | description        | truncated — how badly?       |
| created                | posted_date        | ISO timestamp? verify        |
| id                     | (dedup input)      |                              |

In [0]:
import requests, json, time
from pyspark.sql import functions as F

APP_ID  = dbutils.secrets.get("jobmarket", "adzuna_app_id")
APP_KEY = dbutils.secrets.get("jobmarket", "adzuna_app_key")

SEARCH_TERMS = [
    "data engineer",
    "data analyst",
    "data scientist",
    "machine learning engineer",
    "business intelligence analyst",
    "analytics engineer",
]
PAGES_PER_TERM = 5
BASE_URL = "https://api.adzuna.com/v1/api/jobs/us/search/{page}"

In [0]:
rows = []

for term in SEARCH_TERMS:
    for page in range(1, PAGES_PER_TERM + 1):
        request_id = f"{term.replace(' ', '_')}_p{page}"
        try:
            resp = requests.get(
                BASE_URL.format(page=page),
                params={
                    "app_id": APP_ID,
                    "app_key": APP_KEY,
                    "results_per_page": 50,
                    "what": term,
                },
                timeout=30,
            )
            resp.raise_for_status()
            results = resp.json().get("results", [])
            for posting in results:
                rows.append((json.dumps(posting), "adzuna", request_id))
            print(f"{request_id}: {len(results)} postings")
        except Exception as e:
            print(f"{request_id}: FAILED -> {e}")
        time.sleep(1)

print(f"\nTotal collected: {len(rows)}")

In [0]:
df = (spark.createDataFrame(rows, ["raw_payload", "source", "api_request_id"])
      .withColumn("ingest_ts", F.current_timestamp())
      .withColumn("ingest_date", F.current_date()))

(df.write
   .format("delta")
   .mode("append")
   .partitionBy("ingest_date")
   .saveAsTable("jobmarket.bronze.job_postings_api"))

In [0]:
t = spark.table("jobmarket.bronze.job_postings_api")
today_rows = t.filter(F.col("ingest_date") == F.current_date()).count()

print("Run summary")
print(f"  Collected this run: {len(rows)}")
print(f"  Rows with today's ingest_date: {today_rows}")
print(f"  Bronze API table total: {t.count()}")

if len(rows) == 0:
    raise Exception("INGESTION FAILED: zero postings collected")
print("Ingestion run complete")